In [1]:
!pip install datasets nltk scikit-learn pandas

In [2]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("jquigl/imdb-genres")
df = pd.DataFrame(dataset['train'])

print(df.shape)
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/54.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/238256 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/29809 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/29756 [00:00<?, ? examples/s]

(238256, 5)


,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."
3,Gulliver Returns - 2021,Fantasy,"Animation, Adventure, Family",4.4,The legendary Gulliver returns to the Kingdom ...
4,Prithvi Vallabh - 1924,Biography,"Biography, Drama, Romance",NaN,"Seminal silent historical film, the story feat..."


In [23]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_description'] = df['description'].apply(clean_text)

print("DONE")
df[['clean_description']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


DONE


,clean_description
0,liveaction adaptation jm barries classic tale ...
1,puss boots discovers passion adventure taken t...
2,two longdistance best friends change others li...
4,trip hometown workaholic ally reminisces ex se...
5,seventeenyearold aristocrat falls love kind po...


In [3]:
print('df exists:', 'df' in globals())

df exists: True


In [5]:
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [9]:
import os

path = "/content/archive/"
files = os.listdir(path)

df_list = []

for file in files:
    temp = pd.read_csv(path + file)
    genre = file.replace(".csv", "")
    temp["genre"] = genre
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [10]:
df = df[pd.to_numeric(df["year"], errors="coerce").notnull()]
df["year"] = df["year"].astype(int)

In [11]:
df = df[df["runtime"].notna()]
df["runtime"] = df["runtime"].str.replace("min", "", regex=False)
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")

In [13]:
df["rating"] = pd.to_numeric(df["rating"], errors="coerce").fillna(0)
df["votes"] = pd.to_numeric(df["votes"], errors="coerce").fillna(0)

In [14]:
df["movie_name"] = df["movie_name"].astype(str).str.lower()
df["description"] = df["description"].astype(str)

In [15]:
df = df.drop_duplicates(subset=["movie_name"])

In [16]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["clean_desc"] = df["description"].apply(clean_text)

In [17]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=3
)

tfidf_matrix = tfidf.fit_transform(df["clean_desc"])

print("TF-IDF Shape:", tfidf_matrix.shape)

TF-IDF Shape: (142421, 8000)


In [18]:
indices = pd.Series(df.index, index=df["movie_name"]).drop_duplicates()

In [19]:
def recommend_by_title(title, top_n=5):
    title = title.lower().strip()

    if title not in indices:
        return ["Movie not found"]

    idx = indices[title]

    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # combine similarity + rating + votes
    scores = list(enumerate(sim_scores))
    scores = sorted(
        scores,
        key=lambda x: (
            x[1],
            df.iloc[x[0]]["rating"],
            np.log1p(df.iloc[x[0]]["votes"])
        ),
        reverse=True
    )

    results = []
    for i, _ in scores:
        if i != idx:
            results.append(df.iloc[i]["movie_name"])
        if len(results) == top_n:
            break

    return results

In [20]:
def recommend_by_description(query, top_n=5):
    query_clean = clean_text(query)
    query_vec = tfidf.transform([query_clean])

    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    scores = list(enumerate(sim_scores))
    scores = sorted(
        scores,
        key=lambda x: (
            x[1],
            df.iloc[x[0]]["rating"],
            np.log1p(df.iloc[x[0]]["votes"])
        ),
        reverse=True
    )

    return [df.iloc[i]["movie_name"] for i, _ in scores[:top_n]]

In [21]:
def recommend_by_genre(genre, query, top_n=5):
    filtered = df[df["genre"].str.contains(genre, case=False, na=False)]

    if filtered.empty:
        return ["No movies found"]

    tfidf_local = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    matrix = tfidf_local.fit_transform(filtered["clean_desc"])

    query_vec = tfidf_local.transform([clean_text(query)])
    sim_scores = cosine_similarity(query_vec, matrix).flatten()

    scores = list(enumerate(sim_scores))
    scores = sorted(
        scores,
        key=lambda x: (
            x[1],
            filtered.iloc[x[0]]["rating"]
        ),
        reverse=True
    )

    return [filtered.iloc[i]["movie_name"] for i, _ in scores[:top_n]]

In [22]:
print("\n TITLE TEST:")
print(recommend_by_title("avatar: the way of water"))

print("\n DESCRIPTION TEST:")
print(recommend_by_description("space mission alien planet survival"))

print("\n GENRE TEST:")
print(recommend_by_genre("action", "hero saves world"))


 TITLE TEST:
['the arizona terror', 'poor agnes', 'long distance', 'nu hou kuang hua', 'moss rose']

 DESCRIPTION TEST:
['morpheus one', "project 'gemini'", 'space chimps', 'blizkata dalechina', 'fate federal agent 8']

 GENRE TEST:
['nordexpressen', 'desert raiders', 'terror of the steppes', 'guns of the apocalypse', 'beyond the game']
